In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
# File names
all_events = "momenta_all_events.txt"
delayed_events = "momenta_delayed_events.txt"
direct_events = "momenta_direct_events.txt"

# File used for the first PCA inspection
file = direct_events

In [ ]:
# Column order in the input files
all_columns = [
    "p1x", "p1y", "p1z",
    "p2x", "p2y", "p2z",
    "p3x", "p3y", "p3z",
    "p4x", "p4y", "p4z",
]

# PCA is performed only for the three electrons.
# Particle 1 is the ion/core and is not included in the PCA input.
electron_columns = [
    "p2x", "p2y", "p2z",
    "p3x", "p3y", "p3z",
    "p4x", "p4y", "p4z",
]

# Read the selected dataset
df = pd.read_csv(file, sep=",", header=None, names=all_columns)
df_e = df[electron_columns]

df_e.head()

In [ ]:
# Number of events and PCA input variables
df_e.shape

In [ ]:
# Basic statistics of the electron momentum components
df_e.describe()

In [ ]:
# Missing-value check
# All values should be zero before PCA is performed.
df_e.isna().sum()

In [ ]:
# Standardize every momentum component before PCA.
# StandardScaler gives each input column mean 0 and standard deviation 1,
# so PCA is not dominated only by differences in the numerical scale of columns.
X = df_e.values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit PCA in the full 9-dimensional electron-momentum space.
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

In [ ]:
# Explained variance of every principal component
explained = pca.explained_variance_ratio_
cumulative_explained = np.cumsum(explained)

explained_variance = pd.DataFrame({
    "PC": [f"PC{i + 1}" for i in range(len(explained))],
    "explained_variance_ratio": explained,
    "cumulative_explained_variance": cumulative_explained,
})

explained_variance

In [ ]:
# Explained variance ratio for each principal component
plt.figure(figsize=(8, 5))
plt.plot(np.arange(1, len(explained) + 1), explained, marker="o")
plt.xlabel("Principal component")
plt.ylabel("Explained variance ratio")
plt.title(f"PCA explained variance: {file}")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Cumulative explained variance
plt.figure(figsize=(8, 5))
plt.plot(
    np.arange(1, len(cumulative_explained) + 1),
    cumulative_explained,
    marker="o",
)
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title(f"Cumulative explained variance: {file}")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Projection of the events onto the first two principal components
plt.figure(figsize=(7, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=5, alpha=0.4)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title(f"PCA projection: {file}")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Loadings: coefficients of the original momentum components
# in every principal-component direction.
loadings = pd.DataFrame(
    pca.components_.T,
    index=electron_columns,
    columns=[f"PC{i + 1}" for i in range(len(electron_columns))],
)

loadings

In [ ]:
# The longitudinal z-components are the physically relevant part here,
# because the laser polarization axis is z.
# PC1, PC8 and PC9 are inspected because their loadings are dominated by p2z, p3z and p4z.
longitudinal_components = ["p2z", "p3z", "p4z"]
selected_pcs = ["PC1", "PC8", "PC9"]

loadings.loc[longitudinal_components, selected_pcs]

In [ ]:
# Helper function used to repeat exactly the same PCA procedure
# for another event class without duplicating the analysis code.
def run_pca(file_path):
    data = pd.read_csv(file_path, sep=",", header=None, names=all_columns)
    electron_data = data[electron_columns]

    scaled = StandardScaler().fit_transform(electron_data.values)

    model = PCA()
    scores = model.fit_transform(scaled)

    model_loadings = pd.DataFrame(
        model.components_.T,
        index=electron_columns,
        columns=[f"PC{i + 1}" for i in range(len(electron_columns))],
    )

    model_explained = pd.DataFrame({
        "PC": [f"PC{i + 1}" for i in range(len(electron_columns))],
        "explained_variance_ratio": model.explained_variance_ratio_,
        "cumulative_explained_variance": np.cumsum(model.explained_variance_ratio_),
    })

    return {
        "data": data,
        "electron_data": electron_data,
        "scaled": scaled,
        "scores": scores,
        "model": model,
        "loadings": model_loadings,
        "explained_variance": model_explained,
    }

In [ ]:
# Compare the longitudinal PCA directions for direct and delayed events.
# The sign of a PCA eigenvector is arbitrary, so the important information
# is the relative pattern of coefficients, not the overall sign.
result_direct = run_pca(direct_events)
result_delayed = run_pca(delayed_events)

comparison = pd.concat(
    {
        "direct": result_direct["loadings"].loc[longitudinal_components, selected_pcs],
        "delayed": result_delayed["loadings"].loc[longitudinal_components, selected_pcs],
    },
    axis=1,
)

comparison

In [ ]:
# The PCA result motivates three simple longitudinal directions:
#
# P_col = p2z + p3z + p4z
#     collective longitudinal motion of the three electrons
#
# A3 = p3z - (p2z + p4z) / 2
#     deviation of electron 3 from the average of electrons 2 and 4
#
# R24 = p2z - p4z
#     difference between electrons 2 and 4
#
# These variables are used here as an interpretable approximation
# of the longitudinal subspace identified by PCA.
df_all = pd.read_csv(all_events, sep=",", header=None, names=all_columns)

df_all["P_col"] = df_all["p2z"] + df_all["p3z"] + df_all["p4z"]
df_all["A3"] = df_all["p3z"] - (df_all["p2z"] + df_all["p4z"]) / 2
df_all["R24"] = df_all["p2z"] - df_all["p4z"]

df_all[["p2z", "p3z", "p4z", "P_col", "A3", "R24"]].head()

In [ ]:
# Summary statistics of the three proposed longitudinal variables
summary = df_all[["P_col", "A3", "R24"]].describe().T
summary

In [ ]:
# One-dimensional distributions
for column in ["P_col", "A3", "R24"]:
    plt.figure(figsize=(7, 5))
    plt.hist(df_all[column], bins=80)
    plt.xlabel(column)
    plt.ylabel("Number of events")
    plt.title(f"Distribution of {column}")
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# Pairwise projections of the proposed longitudinal variables
pairs = [
    ("P_col", "A3"),
    ("P_col", "R24"),
    ("A3", "R24"),
]

for x, y in pairs:
    plt.figure(figsize=(6, 6))
    plt.scatter(df_all[x], df_all[y], s=5, alpha=0.3)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.title(f"{x} vs {y}")
    plt.grid(True, alpha=0.3)
    plt.show()